# Feasibility probe (CPU) — can surasan092's DINOv3-S+ arm be reconstructed?

Its `focal_fold*_final.pt` stores `model_state` with a HuggingFace-style DINOv3 backbone
(`backbone.embeddings.*`, `backbone.layer.N.*`, 384-dim, 12 layers, 4 register tokens) plus a
12-tensor head. No inference code was published, so the arm has to be reconstructed — but the
author also published per-fold OOF predictions, which are ground truth to verify against.

This probe answers three questions before any real build:
1. can this environment construct the backbone class and load those 235 tensors cleanly?
2. does a forward pass run on CPU, and how fast?
3. is `1152 = 3 x 384` consistent with concat(CLS, mean-patch, top-12.5%-patch)?

In [ ]:
import torch, json, os, sys, time
import transformers
print('transformers', transformers.__version__, '| torch', torch.__version__)
CK = None
for r,d,f in os.walk('/kaggle/input'):
    d[:] = [x for x in d if x not in ('train_series','test_series')]
    if 'focal_fold0_final.pt' in f: CK = os.path.join(r,'focal_fold0_final.pt'); break
print('checkpoint:', CK)
ck = torch.load(CK, map_location='cpu', weights_only=False)
sd = ck['model_state']
print('hf_model_name:', ck.get('hf_model_name'))
print('tensors:', len(sd))
head = {k:tuple(v.shape) for k,v in sd.items() if not k.startswith('backbone')}
print('head:', json.dumps(head, indent=1))

In [ ]:
# Q1: construct the backbone exactly. HF repos are gated, so build the config from the
# spec the checkpoint itself implies: 384-dim, 12 layers, patch 16, 4 register tokens,
# SwiGLU MLP (gate+up+down) with intermediate 1536, k_proj without bias.
import inspect
from transformers import AutoModel
from transformers.models.dinov3_vit.configuration_dinov3_vit import DINOv3ViTConfig
from transformers.models.dinov3_vit.modeling_dinov3_vit import DINOv3ViTModel
sig = inspect.signature(DINOv3ViTConfig.__init__)
print('DINOv3ViTConfig params:'); print(' ', sorted(sig.parameters))
bst = {k[len('backbone.'):]: v for k,v in sd.items() if k.startswith('backbone.')}

base = dict(hidden_size=384, num_hidden_layers=12, num_attention_heads=6,
            patch_size=16, num_register_tokens=4, image_size=336,
            intermediate_size=1536)
ok = False
for extra in [{}, {'use_gated_mlp': True}, {'mlp_ratio': 4.0, 'use_gated_mlp': True},
              {'hidden_act':'silu','use_gated_mlp':True}]:
    kw = {k:v for k,v in {**base, **extra}.items() if k in sig.parameters}
    try:
        cfg = DINOv3ViTConfig(**kw)
        bb = DINOv3ViTModel(cfg)
        missing, unexpected = bb.load_state_dict(bst, strict=False)
        print(f'\n  extra={extra} -> missing {len(missing)} unexpected {len(unexpected)}')
        if unexpected[:3]: print('    unexpected:', unexpected[:3])
        if missing[:3]:    print('    missing   :', missing[:3])
        if not missing and not unexpected:
            ok = True; print('    EXACT MATCH'); break
    except Exception as e:
        print(f'\n  extra={extra} -> {type(e).__name__}: {str(e)[:110]}')
print('\nQ1 backbone loadable exactly:', ok)


In [ ]:
# Q2: does a forward pass run on CPU, and how fast? Q3: does 3x384 line up?
if ok:
    bb.eval()
    x = torch.randn(2, 3, 336, 336)
    t0 = time.time()
    with torch.no_grad(): out = bb(pixel_values=x)
    dt = time.time() - t0
    h = out.last_hidden_state
    print('last_hidden_state', tuple(h.shape), f'| {dt/2*1000:.0f} ms/image on CPU')
    n_prefix = 1 + int(cfg.num_register_tokens)
    patches = h[:, n_prefix:]
    print(f'  prefix tokens {n_prefix} (cls + registers) | patch tokens {patches.shape[1]} '
          f'= ({336//16})^2 = {(336//16)**2}')
    cls = h[:,0]; mean = patches.mean(1)
    k = max(1, int(round(0.125 * patches.shape[1])))
    focal = patches.topk(k, dim=1).values.mean(1)   # hypothesis for the 3rd part
    feat = torch.cat([cls, mean, focal], -1)
    print(f'  concat(cls, mean, top-{k}) -> {tuple(feat.shape)}; feature_proj expects '
          f'{sd["feature_proj.weight"].shape[1]}')
    print('  SHAPES MATCH:', feat.shape[-1] == sd['feature_proj.weight'].shape[1])
    est = dt/2 * 58 * 6 * 12 / 60
    print(f'\n  est CPU cost to score the 58 gold studies (6 slots x 12 slices): {est:.0f} min/fold')